# examples-seen-step-axis — worked example 2: examples_seen with Gradient Accumulation

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `examples-seen-step-axis`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

When using gradient accumulation with `accum_steps` micro-batches per optimizer step, each optimizer step has processed `batch_size * accum_steps` real examples. The correct formula is `examples_seen = optimizer_step * batch_size * accum_steps`. This ensures that a run with batch_size=8, accum_steps=4 plots identically to a run with batch_size=32, accum_steps=1 — they have the same effective batch size.

## Worked solution

We compare two configurations with the same effective batch size (32) but different micro-batch / accumulation splits.

**Config A:** batch_size=32, accum_steps=1 (standard). After 5 optimizer steps: 32, 64, 96, 128, 160 examples seen.

**Config B:** batch_size=8, accum_steps=4 (gradient accumulation). After 5 optimizer steps: 8*4=32, 64, 96, 128, 160 examples seen.

**The formula:** `examples_seen = opt_step * batch_size * accum_steps`. The per-step contribution is always `batch_size * accum_steps = effective_batch_size`.

Since both configs have the same effective batch size, their `examples_seen` sequences are identical — plots will perfectly overlap. Without the accum_steps factor, config B's curve would be 4× shifted to the left, misleadingly appearing to learn faster.

In [ ]:
import torch as t

def examples_seen_with_accum(opt_step, batch_size, accum_steps):
    return opt_step * batch_size * accum_steps

# Config A: standard, bs=32, accum=1
config_a = [(s, examples_seen_with_accum(s, 32, 1)) for s in range(1, 6)]
# Config B: accumulated, bs=8, accum=4
config_b = [(s, examples_seen_with_accum(s, 8, 4)) for s in range(1, 6)]

print("Config A (bs=32, accum=1):  ", [ex for _, ex in config_a])
print("Config B (bs=8,  accum=4):  ", [ex for _, ex in config_b])

# They must be identical step-by-step
for (sa, exa), (sb, exb) in zip(config_a, config_b):
    assert exa == exb, f"Mismatch at step {sa}: {exa} vs {exb}"

print("Confirmed: same effective batch size => identical examples_seen curves.")